# Лабораторна робота №2. Частина 1: 
**Виконав:** Студент групи ФБ-46 — Ільченко Влад

In [ ]:
import os
import pandas as pd
import numpy as np

def load_clean_dataframe(folder_path):
    """
    Зчитує дані за допомогою pd.read_csv (без ручного open()), 
    очищує колонки та додає ідентифікатор області.
    """
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Папку '{folder_path}' не знайдено локально.")
        
    all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    combined_data = []
    
    for i, file_name in enumerate(all_files):
        file_path = os.path.join(folder_path, file_name)
        
        # Викладач вимагав pd.read_csv. Пропускаємо сміттєвий заголовок NOAA (skiprows=1)
        temp_df = pd.read_csv(file_path, skiprows=1, index_col=False)
        
        # Очищаємо назви колонок від пробілів
        temp_df.columns = [c.replace(' ', '').strip() for c in temp_df.columns]
        
        # Видаляємо порожні або повністю зашумлені рядки, які бувають в кінці файлів NOAA
        temp_df.dropna(subset=['Year', 'Week', 'VHI'], inplace=True)
        
        # Додаємо ID області (наприклад, від 1 до 27)
        temp_df['Area_ID'] = i + 1
        
        # Приводимо типи даних до чистих числових
        temp_df['Year'] = temp_df['Year'].astype(int)
        temp_df['Week'] = temp_df['Week'].astype(int)
        
        combined_data.append(temp_df)
    
    # Об'єднуємо всі файли в один великий DataFrame
    full_df = pd.concat(combined_data, ignore_index=True)
    return full_df

# Ініціалізація та запуск зчитування
folder = "vhi_data"
try:
    df = load_clean_dataframe(folder)
    print(f"=== ДАНІ УСПІШНО ЗАВАНТАЖЕНО ЗА ДОПОМОГОЮ PD.READ_CSV ===")
    print(f"Загальна кількість рядків: {len(df)}")
except Exception as e:
    print(f"Помилка: {e}. Переконайся, що папка '{folder}' є поруч з ноутбуком.")

### Завдання 1. Вивести екстремуми (мінімум та максимум) індексу VHI для заданої області за конкретний рік.

In [ ]:
def get_vhi_extremes(dataframe, area_id, year):
    """Повертає мінімальне та максимальне значення VHI для обраної області та року"""
    filtered = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['Year'] == year)]
    
    if filtered.empty:
        return None, None
        
    min_vhi = filtered['VHI'].min()
    max_vhi = filtered['VHI'].max()
    return min_vhi, max_vhi

In [ ]:
# Тестовий виклик для Області №9 (Київська) за 2020 рік
target_area = 9
target_year = 2020

min_v, max_v = get_vhi_extremes(df, target_area, target_year)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 1:")
if min_v is not None:
    print(f"Область ID: {target_area}, Рік: {target_year}")
    print(f"-> Мінімальний індекс VHI: {min_v}")
    print(f"-> Максимальний індекс VHI: {max_v}")
else:
    print(f"Даних для Області ID {target_area} за {target_year} рік не знайдено.")

### Завдання 2. Знайти роки, в які спостерігалися сильні та екстремальні посухи (VHI менше заданого порогу) для обраної області.

In [ ]:
def find_drought_years(dataframe, area_id, vhi_threshold=20.0):
    """Повертає список унікальних років, коли VHI падав нижче критичного порогу"""
    filtered = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['VHI'] < vhi_threshold)]
    unique_years = sorted(filtered['Year'].unique())
    return unique_years

In [ ]:
# Шукаємо роки з екстремальною посухою (VHI < 15) для Області №12 (Львівська)
area_test = 12
threshold_test = 15.0

drought_years = find_drought_years(df, area_test, threshold_test)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 2:")
print(f"Роки з екстремальною посухою (VHI < {threshold_test}) для області ID {area_test}:")
print(drought_years)

### Завдання 3. Фільтрація всього датасету за умовою: вивести роки та тижні, де індекс VHI знаходився в межах помірної посухи (від 20 до 35).

In [ ]:
def filter_moderate_drought(dataframe):
    """Фільтрує записи, де VHI знаходиться в межах від 20 до 35 включно"""
    result_df = dataframe[dataframe['VHI'].between(20.0, 35.0)]
    return result_df[['Year', 'Week', 'VHI', 'Area_ID']]

In [ ]:
# Отримуємо відфільтровану таблицю
moderate_drought_data = filter_moderate_drought(df)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 3:")
print(f"Всього знайдено записів з помірною посухою: {len(moderate_drought_data)}")
# Показуємо перші 10 рядків результату у вигляді красивої таблиці Pandas
moderate_drought_data.head(10)